<a href="https://colab.research.google.com/github/TomaRaul/Comparative_Study/blob/main/dinoVSresnet21_01_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch torchvision timm albumentations tqdm

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models.segmentation import deeplabv3_resnet50
import timm
import numpy as np
from tqdm import tqdm

In [ ]:
import torchvision.transforms.functional as TF
import random

# Funcție pentru transformări sincronizate (imagine + mască)
def custom_transforms(image, mask):
    # 1. Redimensionare la 518x518 (optim pentru DINOv2 patch 14)
    image = TF.resize(image, (518, 518))
    mask = TF.resize(mask, (518, 518), interpolation=transforms.InterpolationMode.NEAREST)

    # 2. Random Horizontal Flip (simultan)
    if random.random() > 0.5:
        image = TF.hflip(image)
        mask = TF.hflip(mask)

    # 3. Conversie Image -> Tensor + Normalizare ImageNet
    image = TF.to_tensor(image)
    image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    # 4. Conversie Mască -> Tensor de numere întregi (Long)
    # IMPORTANT: Nu folosim to_tensor() aici pentru a nu scala valorile la 0-1
    mask = torch.from_numpy(np.array(mask)).long()

    # 5. Curățare Mască Oxford: 1=Animal, 2=Fundal, 3=Contur
    # Transformăm în: 1=Animal, 0=Restul (Semantic Segmentation)
    final_mask = torch.zeros_like(mask)
    final_mask[mask == 1] = 1

    return image, final_mask

# Helper pentru a integra transformările în obiectul Dataset
class PetDataset(datasets.OxfordIIITPet):
    def __getitem__(self, index):
        img, mask = super().__getitem__(index)
        return custom_transforms(img, mask)

# Re-inițializare Datasets
train_ds = PetDataset(root='data', split='trainval', target_types='segmentation', download=True)
val_ds   = PetDataset(root='data', split='test', target_types='segmentation', download=True)

# DataLoader
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=8)

print("Dataset configurat corect pentru segmentare semantică!")

In [ ]:
model_resnet = deeplabv3_resnet50(pretrained=True)
model_resnet.classifier[4] = nn.Conv2d(256, 2, 1) # 2 clase: fundal + animal

In [ ]:
import torch
import torch.nn as nn
import timm

# CORECTARE: Am eliminat '.cuda' în plus
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Re-încărcăm backbone-ul pentru a fi siguri
backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m', pretrained=True, features_only=True)

class DinoSeg(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        # Decoder cu mai multe straturi pentru detalii fine
        self.decoder = nn.Sequential(
            nn.Conv2d(768, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 2, kernel_size=1) # Ieșire: 2 clase (Background, Pet)
        )

    def forward(self, x):
        # Extragem trăsăturile din ultimul strat al Transformer-ului
        # DINOv2 returnează o listă de trăsături; [-1] este cea mai profundă
        feat = self.backbone(x)[-1]

        # Trecem prin decoder-ul convoluțional
        out = self.decoder(feat)

        # Upsampling la mărimea originală a imaginii (518x518)
        return nn.functional.interpolate(out, size=x.shape[2:], mode='bilinear', align_corners=False)

# Instanțiere și trimitere pe GPU/CPU
model_dino = DinoSeg(backbone).to(device)
print(f"Model DINOv2 configurat cu succes pe dispozitivul: {device}")

In [ ]:
def train(model, loader, optimizer):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0

    for imgs, masks in tqdm(loader):
        imgs = imgs.to(device)
        masks = masks.squeeze(1).long().to(device)

        out = model(imgs)

        if isinstance(out, dict):     # pentru DeepLab
            preds = out['out']
        else:                         # pentru DINO
            preds = out

        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [ ]:
def evaluate(model, loader):
    model.eval()
    ious = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)

            # masks: [B,1,H,W] -> [B,H,W]
            masks = masks.squeeze(1).long().to(device)

            out = model(imgs)

            if isinstance(out, dict):
                preds = out['out']
            else:
                preds = out

            # [B,C,H,W] -> [B,H,W]
            preds = torch.argmax(preds, dim=1)

            # CLASA animal = 1
            pred_fg  = (preds == 1)
            mask_fg  = (masks == 1)

            inter = (pred_fg & mask_fg).sum(dim=(1,2))
            union = (pred_fg | mask_fg).sum(dim=(1,2))

            iou = (inter + 1e-6) / (union + 1e-6)
            ious.extend(iou.cpu().numpy())

    return float(np.mean(ious))


In [ ]:
# Setare dispozitiv de calcul (GPU daca e disponibil, altfel CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_resnet.to(device)
model_dino.to(device)

EPOCHS = 10
UNFREEZE_EPOCH = 1 # Epoca la care se dezgheata restul retelei

# Faza 1: Antrenam doar straturile de segmentare (backbone-ul ramane inghetat)
# Folosim filter pentru a selecta doar parametrii care necesita actualizare
opt_resnet = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_resnet.parameters()),
    lr=1e-4, weight_decay=1e-4
)

opt_dino = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_dino.parameters()),
    lr=1e-4, weight_decay=1e-4
)

# Liste pentru stocarea istoricului de antrenament pentru grafice
resnet_loss_hist = []
dino_loss_hist = []
resnet_iou_hist = []
dino_iou_hist = []

unfrozen = False   # Flag pentru a marca trecerea la faza de fine-tuning total

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch}")

    # Sectiunea de antrenare (Training)
    model_resnet.train()
    l_resnet = train(model_resnet, train_loader, opt_resnet)
    torch.cuda.empty_cache() # Curatare memorie video dupa fiecare model

    model_dino.train()
    l_dino = train(model_dino, train_loader, opt_dino)
    torch.cuda.empty_cache()

    # Sectiunea de evaluare (Validation)
    model_resnet.eval()
    model_dino.eval()

    # Calculam metrica IoU pe setul de validare
    iou_resnet = evaluate(model_resnet, val_loader)
    iou_dino = evaluate(model_dino, val_loader)

    # Inregistrarea datelor pentru monitorizare
    resnet_loss_hist.append(l_resnet)
    dino_loss_hist.append(l_dino)
    resnet_iou_hist.append(iou_resnet)
    dino_iou_hist.append(iou_dino)

    print(f"ResNet Loss: {l_resnet:.4f} | IoU: {iou_resnet:.4f}")
    print(f"DINO   Loss: {l_dino:.4f} | IoU: {iou_dino:.4f}")

    # Faza 2: Activarea Fine-tuning-ului total pentru toata reteaua
    if epoch == UNFREEZE_EPOCH and not unfrozen:
        unfrozen = True
        print(">> Unfreezing backbones")

        # Activam calculul gradientilor pentru backbone-ul ResNet
        for p in model_resnet.backbone.parameters():
            p.requires_grad = True

        # Re-initializam optimizatorul cu rate de invatare diferite pentru stabilitate
        opt_resnet = torch.optim.AdamW([
            {'params': model_resnet.backbone.parameters(), 'lr': 1e-5}, # LR mic pt backbone
            {'params': model_resnet.classifier.parameters(), 'lr': 1e-4} # LR mare pt capat
        ], weight_decay=1e-4)

        # Activam calculul gradientilor pentru backbone-ul DINO
        for p in model_dino.backbone.parameters():
            p.requires_grad = True

        # Aplicam rate de invatare diferentiate pentru Transformer
        opt_dino = torch.optim.AdamW([
            {'params': model_dino.backbone.parameters(), 'lr': 1e-5},
            {'params': model_dino.decoder.parameters(),  'lr': 1e-4}
        ], weight_decay=1e-4)

In [ ]:
import matplotlib.pyplot as plt

epochs = range(10)

# LOSS
plt.figure()
plt.plot(epochs, resnet_loss_hist, label='ResNet Loss')
plt.plot(epochs, dino_loss_hist, label='DINO Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Comparison")
plt.legend()
plt.show()

# IoU
plt.figure()
plt.plot(epochs, resnet_iou_hist, label='ResNet IoU')
plt.plot(epochs, dino_iou_hist, label='DINO IoU')
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.title("IoU Comparison")
plt.legend()
plt.show()


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "epoch": list(range(1, EPOCHS + 1)),
    "resnet_loss": resnet_loss_hist,
    "dino_loss": dino_loss_hist,
    "resnet_iou": resnet_iou_hist,
    "dino_iou": dino_iou_hist
})

df.to_csv("training_results.csv", index=False)
print("✔️ Datele au fost salvate în training_results.csv")

plt.savefig("loss_comparison.png", dpi=300)
plt.show()

plt.savefig("iou_comparison.png", dpi=300)
plt.show()

In [ ]:
# Salvare modele antrenate
torch.save(model_resnet.state_dict(), 'model_resnet_final.pth')
torch.save(model_dino.state_dict(), 'model_dino_final.pth')
print("Modelele au fost salvate cu succes!")
print(f" model_resnet_final.pth")
print(f" model_dino_final.pth")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import torch

def visualize_model_comparison(index=0):
    # Luăm o imagine din setul de validare (val_ds)
    img_tensor, mask_truth = val_ds[index]

    # Punem modelele în modul de evaluare
    model_resnet.eval()
    model_dino.eval()

    # Pregătim imaginea pentru modele (batch de 1 + trimitere pe GPU/device)
    input_batch = img_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        # Predicție ResNet (DeepLab returnează un dicționar)
        out_res = model_resnet(input_batch)['out']
        pred_res = out_res.argmax(1).squeeze().cpu().numpy()

        # Predicție DINO
        out_dino = model_dino(input_batch)
        pred_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    # Pregătire imagine pentru afișare (de-normalizare inversă)
    display_img = img_tensor.permute(1, 2, 0).cpu().numpy()
    # Folosim valorile standard ImageNet pentru a readuce culorile la normal
    display_img = (display_img * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
    display_img = np.clip(display_img, 0, 1)

    # Plotare rezultate (4 coloane pentru comparație tip 1)
    plt.figure(figsize=(20, 5))

    titles = ['Imagine Originală', 'Masca Reală (GT)', 'Predicție ResNet50', 'Predicție DINOv2']
    # Masca reală trebuie convertită în numpy pentru afișare
    imgs = [display_img, mask_truth.cpu().numpy(), pred_res, pred_dino]

    for i in range(4):
        plt.subplot(1, 4, i+1)
        # Pentru măști folosim harta de culori 'viridis' (ca în cerința ta)
        cmap = 'viridis' if i > 0 else None
        plt.imshow(imgs[i], cmap=cmap)
        plt.title(titles[i])
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Apelează funcția pentru a vedea rezultatul comparativ
# Poți schimba indexul (ex: 15, 40, 100) pentru a vedea animale diferite
visualize_model_comparison(index=20)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import os

# 1. Definim transformarile necesare pentru procesarea imaginilor
transform_img = transforms.Compose([
    # Redimensionare la 518x518 (optim pentru patch-urile de 14 ale DINOv2)
    transforms.Resize((518, 518)),
    transforms.ToTensor(),
    # Normalizare folosind media si deviatia standard ImageNet
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Functie pentru predictia segmentarii folosind un singur model
def predict_segmentation(image_path, model, model_name):
    # Verificam daca fisierul exista la calea specificata
    if not os.path.exists(image_path):
        print(f"Imaginea {image_path} nu exista.")
        return

    # Incarcam imaginea si o pregatim pentru model
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    # Trecem modelul in modul de evaluare
    model.eval()
    with torch.no_grad():
        out = model(input_tensor)
        # Gestionam diferentele de output intre DeepLab si DINO
        preds = out['out'] if isinstance(out, dict) else out
        # Extragem clasa cu probabilitatea maxima pentru fiecare pixel
        mask = preds.argmax(1).squeeze().cpu().numpy()

    # Afisam imaginea originala si masca rezultata
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img_pil.resize((518, 518)))
    plt.title(f"Original ({model_name})")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap='viridis')
    plt.title(f"Masca {model_name}")
    plt.axis('off')
    plt.show()

# 3. Functie pentru comparatia directa intre cele doua modele
def compare_models(image_path):
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    # Pregatim ambele modele pentru inferenta
    model_resnet.eval()
    model_dino.eval()

    with torch.no_grad():
        # Obtinem predictia pentru modelul ResNet
        out_res = model_resnet(input_tensor)['out']
        mask_res = out_res.argmax(1).squeeze().cpu().numpy()

        # Obtinem predictia pentru modelul DINO
        out_dino = model_dino(input_tensor)
        mask_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    # Generam graficul comparativ cu 3 coloane
    plt.figure(figsize=(15, 5))
    imgs = [img_pil.resize((518, 518)), mask_res, mask_dino]
    titles = ["Original", "ResNet", "DINOv2"]
    for i in range(3):
        plt.subplot(1, 3, i+1)
        # Aplicam colormap-ul viridis doar pentru mastile de segmentare
        plt.imshow(imgs[i], cmap='viridis' if i > 0 else None)
        plt.title(titles[i])
        plt.axis('off')
    plt.show()

print("Toate functiile de vizualizare sunt gata!")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Functie pentru realizarea predictiei si afisarea rezultatului pentru un model
def predict_segmentation(image_path, model, model_name):
    # Incarcarea imaginii de pe disc si conversia la format RGB
    img_pil = Image.open(image_path).convert('RGB')

    # Aplicarea transformarilor si pregatirea tensorului pentru GPU/CPU
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    # Setarea modelului in mod evaluare pentru a opri Dropout/BatchNorm
    model.eval()
    with torch.no_grad():
        # Obtinerea predictiei brute de la model
        out = model(input_tensor)
        # Gestionarea diferentelor de structura: DeepLab (dict) vs DINO (tensor)
        preds = out['out'] if isinstance(out, dict) else out
        # Conversia scorurilor in masca finala (0 sau 1) pentru fiecare pixel
        mask = preds.argmax(1).squeeze().cpu().numpy()

    # Crearea graficului comparativ: Imagine Originala vs Masca Prezisa
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    # Redimensionare doar pentru afisare vizuala consistenta
    plt.imshow(img_pil.resize((518, 518)))
    plt.title(f"Original ({model_name})")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    # Afisarea mastii folosind colormap-ul viridis (galben/mov)
    plt.imshow(mask, cmap='viridis')
    plt.title(f"Masca {model_name}")
    plt.axis('off')
    plt.show()

# Functie pentru vizualizarea paralela a rezultatelor ambelor modele
def compare_models(image_path):
    # Pregatirea imaginii de test
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    # Dezactivarea antrenamentului pentru ambele retele
    model_resnet.eval()
    model_dino.eval()

    with torch.no_grad():
        # Predictie pentru arhitectura CNN (ResNet)
        out_res = model_resnet(input_tensor)['out']
        mask_res = out_res.argmax(1).squeeze().cpu().numpy()

        # Predictie pentru arhitectura Transformer (DINOv2)
        out_dino = model_dino(input_tensor)
        mask_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    # Afisare pe 3 coloane pentru comparatie directa tip 1
    plt.figure(figsize=(15, 5))
    imgs = [img_pil.resize((518, 518)), mask_res, mask_dino]
    titles = ["Imagine Originala", "Predictie ResNet", "Predictie DINOv2"]

    for i in range(3):
        plt.subplot(1, 3, i+1)
        # Aplicarea hartii de culori doar pe mastile de segmentare
        plt.imshow(imgs[i], cmap='viridis' if i > 0 else None)
        plt.title(titles[i])
        plt.axis('off')
    plt.show()

print("Functiile de testare au fost definite cu succes!")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
import os

# 1. Definirea pipeline-ului de preprocesare a imaginilor
transform_img = transforms.Compose([
    # Redimensionare la 518x518 pentru compatibilitate cu patch size 14 din DINOv2
    transforms.Resize((518, 518)),
    # Conversia imaginii in tensor PyTorch
    transforms.ToTensor(),
    # Normalizarea standard ImageNet necesara pentru modelele preantrenate
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Functie pentru realizarea predictiei pe un singur model selectat
def predict_segmentation(image_path, model, model_name):
    # Verificarea existentei fisierului la calea specificata
    if not os.path.exists(image_path):
        print(f"Imaginea {image_path} nu exista.")
        return

    # Incarcarea imaginii si pregatirea tensorului de intrare
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    # Setarea modelului in modul evaluare (opreste Dropout si BatchNorm)
    model.eval()
    with torch.no_grad():
        out = model(input_tensor)
        # Gestionarea formatului de iesire: dict pentru DeepLab, tensor pentru DINO
        preds = out['out'] if isinstance(out, dict) else out
        # Selectarea clasei cu scorul cel mai mare pentru fiecare pixel (0 sau 1)
        mask = preds.argmax(1).squeeze().cpu().numpy()

    # Vizualizarea rezultatului individual
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img_pil.resize((518, 518)))
    plt.title(f"Original ({model_name})")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    # Aplicarea hartii de culori viridis (galben pentru obiect, mov pentru fundal)
    plt.imshow(mask, cmap='viridis')
    plt.title(f"Masca {model_name}")
    plt.axis('off')
    plt.show()

# 3. Functie pentru comparatia vizuala directa intre ResNet si DINOv2
def compare_models(image_path):
    img_pil = Image.open(image_path).convert('RGB')
    input_tensor = transform_img(img_pil).unsqueeze(0).to(device)

    # Dezactivarea calculului de gradienti pentru ambele modele
    model_resnet.eval()
    model_dino.eval()

    with torch.no_grad():
        # Obtinerea mastii de la modelul bazat pe ResNet
        out_res = model_resnet(input_tensor)['out']
        mask_res = out_res.argmax(1).squeeze().cpu().numpy()

        # Obtinerea mastii de la modelul bazat pe DINOv2
        out_dino = model_dino(input_tensor)
        mask_dino = out_dino.argmax(1).squeeze().cpu().numpy()

    # Generarea plotului comparativ pe 3 coloane
    plt.figure(figsize=(15, 5))
    imgs = [img_pil.resize((518, 518)), mask_res, mask_dino]
    titles = ["Original", "ResNet", "DINOv2"]
    for i in range(3):
        plt.subplot(1, 3, i+1)
        # Afisarea mastilor cu harta de culori viridis pentru contrast optim
        plt.imshow(imgs[i], cmap='viridis' if i > 0 else None)
        plt.title(titles[i])
        plt.axis('off')
    plt.show()

print("Toate functiile de vizualizare sunt gata!")

In [ ]:
# Test pe câteva imagini din dataset
import os

# Găsește câteva imagini de test
image_dir = '/content/data/oxford-iiit-pet/images'
test_images = [
    'Abyssinian_1.jpg',
    'american_bulldog_146.jpg',
    'Birman_173.jpg',
    'Bengal_100.jpg'
]

print("🔍 Testare modele pe imagini sample...\n")

for img_name in test_images:
    img_path = os.path.join(image_dir, img_name)

    if os.path.exists(img_path):
        print(f"\n{'='*60}")
        print(f" Testare pe: {img_name}")
        print(f"{'='*60}")

        # Test ResNet
        print("\n ResNet:")
        predict_segmentation(img_path, model_resnet, "ResNet")

        # Test DINO
        print("\n DINO:")
        predict_segmentation(img_path, model_dino, "DINO")

        # Comparație directă
        print("\n Comparație directă:")
        compare_models(img_path)
    else:
        print(f"⚠️ Imaginea {img_name} nu a fost găsită")

print("\n Testare completă!")